**EDA And Data Preprocessing**

Import Data

In [ ]:
import pandas as pd
data = pd.read_csv('twitter_training.csv',names=['ID', 'Topic', 'Sentiment', 'Tweet'])

In [ ]:
val_data = pd.read_csv('twitter_validation.csv',names=['ID', 'Topic', 'Sentiment', 'Tweet'])

Display Data Info

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74682 entries, 0 to 74681
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   ID         74682 non-null  int64 
 1   Topic      74682 non-null  object
 2   Sentiment  74682 non-null  object
 3   Tweet      73996 non-null  object
dtypes: int64(1), object(3)
memory usage: 2.3+ MB


In [ ]:
data['Sentiment'].value_counts()

,count
Sentiment,
Negative,22542
Positive,20832
Neutral,18318
Irrelevant,12990


In [ ]:
data.isnull().sum()

,0
ID,0
Topic,0
Sentiment,0
Tweet,686


Clean up missing values

In [ ]:
data = data.dropna()
data.isnull().sum()

,0
ID,0
Topic,0
Sentiment,0
Tweet,0


Delete irrelevant columns

In [ ]:
data.drop(['Topic' , 'ID'] , axis=1 , inplace = True)

<ipython-input-8-be6741261a33>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data.drop(['Topic' , 'ID'] , axis=1 , inplace = True)


In [ ]:
val_data.drop(['Topic' , 'ID'] , axis=1 , inplace = True)

Define the clean text function

In [ ]:
import re
import string
import nltk

nltk.download('punkt_tab')
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

chat_words = {
    "AFAIK": "As Far As I Know",
    "AFK": "Away From Keyboard",
    "ASAP": "As Soon As Possible",
    "ATK": "At The Keyboard",
    "ATM": "At The Moment",
    "A3": "Anytime, Anywhere, Anyplace",
    "BAK": "Back At Keyboard",
    "BBL": "Be Back Later",
    "BBS": "Be Back Soon",
    "BFN": "Bye For Now",
    "B4N": "Bye For Now",
    "BRB": "Be Right Back",
    "BRT": "Be Right There",
    "BTW": "By The Way",
    "B4": "Before",
    "CU": "See You",
    "CUL8R": "See You Later",
    "CYA": "See You",
    "FAQ": "Frequently Asked Questions",
    "FC": "Fingers Crossed",
    "FWIW": "For What It's Worth",
    "FYI": "For Your Information",
    "GAL": "Get A Life",
    "GG": "Good Game",
    "GN": "Good Night",
    "GMTA": "Great Minds Think Alike",
    "GR8": "Great!",
    "G9": "Genius",
    "IC": "I See",
    "ICQ": "I Seek you (also a chat program)",
    "ILU": "ILU: I Love You",
    "IMHO": "In My Honest/Humble Opinion",
    "IMO": "In My Opinion",
    "IOW": "In Other Words",
    "IRL": "In Real Life",
    "KISS": "Keep It Simple, Stupid",
    "LDR": "Long Distance Relationship",
    "LMAO": "Laugh My A.. Off",
    "LOL": "Laughing Out Loud",
    "LTNS": "Long Time No See",
    "L8R": "Later",
    "MTE": "My Thoughts Exactly",
    "M8": "Mate",
    "NRN": "No Reply Necessary",
    "OIC": "Oh I See",
    "PITA": "Pain In The A..",
    "PRT": "Party",
    "PRW": "Parents Are Watching",
    "QPSA?": "Que Pasa?",
    "ROFL": "Rolling On The Floor Laughing",
    "ROFLOL": "Rolling On The Floor Laughing Out Loud",
    "ROTFLMAO": "Rolling On The Floor Laughing My A.. Off",
    "SK8": "Skate",
    "STATS": "Your sex and age",
    "ASL": "Age, Sex, Location",
    "THX": "Thank You",
    "TTFN": "Ta-Ta For Now!",
    "TTYL": "Talk To You Later",
    "U": "You",
    "U2": "You Too",
    "U4E": "Yours For Ever",
    "WB": "Welcome Back",
    "WTF": "What The F...",
    "WTG": "Way To Go!",
    "WUF": "Where Are You From?",
    "W8": "Wait...",
    "7K": "Sick:-D Laugher",
    "TFW": "That feeling when",
    "MFW": "My face when",
    "MRW": "My reaction when",
    "IFYP": "I feel your pain",
    "TNTL": "Trying not to laugh",
    "JK": "Just kidding",
    "IDC": "I don't care",
    "ILY": "I love you",
    "IMU": "I miss you",
    "ADIH": "Another day in hell",
    "ZZZ": "Sleeping, bored, tired",
    "WYWH": "Wish you were here",
    "TIME": "Tears in my eyes",
    "BAE": "Before anyone else",
    "FIMH": "Forever in my heart",
    "BSAAW": "Big smile and a wink",
    "BWL": "Bursting with laughter",
    "BFF": "Best friends forever",
    "CSL": "Can't stop laughing",
    "L8": "Late",
    "SMH": "Shaking My Head",
    "YOLO": "You Only Live Once",
    "TLDR": "Too Long; Didn't Read",
    "FOMO": "Fear Of Missing Out",
    "IDK": "I Don't Know",
    "BFFL": "Best Friends For Life",
    "TMI": "Too Much Information",
    "DM": "Direct Message",
    "STFU": "Shut The F... Up",
    "WTH": "What The Heck",
    "LMAOROTF": "Laughing My A... Off Rolling On The Floor",
    "PPL": "People",
    "SFLR": "Sorry For Late Reply",
    "G2G": "Got To Go",
    "S2R": "Send To Receive"
}


def replace_chat_words(text, chat_words):
    pattern = re.compile(r'\b(' + '|'.join(map(re.escape, chat_words.keys())) + r')\b', flags=re.IGNORECASE)
    def replacer(match):
        word = match.group(0)
        return chat_words.get(word.upper(), word)
    return pattern.sub(replacer, text)

def remove_emojis(text):
    emoji_pattern = re.compile("["
                    u"\U0001F600-\U0001F64F"
                    u"\U0001F300-\U0001F5FF"
                    u"\U0001F680-\U0001F6FF"
                    u"\U0001F1E0-\U0001F1FF"
                    "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)


def clean_text(text, chat_words):
    text = text.lower()

    text = re.sub(r'<[^>]+>', '', text)

    text = re.sub(r'http\S+|www\.\S+', '', text)

    text = replace_chat_words(text, chat_words)

    text = text.translate(str.maketrans('', '', string.punctuation))

    tokens = word_tokenize(text)

    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]

    tokens = [remove_emojis(word) for word in tokens]

    text = " ".join(tokens)
    text = re.sub(r'\s+', ' ', text).strip()

    lemmatizer = WordNetLemmatizer()
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    cleaned_text = " ".join(tokens)
    return cleaned_text

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


Clean text content

In [ ]:
data['Clean_tweet'] = data['Tweet'].apply(lambda x: clean_text(x, chat_words))

val_data['Clean_tweet'] = val_data['Tweet'].apply(lambda x: clean_text(x, chat_words))

print(data[['Tweet', 'Clean_tweet']].head())


                                               Tweet  \
0  im getting on borderlands and i will murder yo...   
1  I am coming to the borders and I will kill you...   
2  im getting on borderlands and i will kill you ...   
3  im coming on borderlands and i will murder you...   
4  im getting on borderlands 2 and i will murder ...   

                      Clean_tweet  
0    im getting borderland murder  
1              coming border kill  
2      im getting borderland kill  
3     im coming borderland murder  
4  im getting borderland 2 murder  


Split the dataset

In [ ]:
X_train = data['Clean_tweet']
Y_train = data['Sentiment']

# Splitting the data into X_test and y_test
X_test = val_data['Clean_tweet']
Y_test = val_data['Sentiment']

 Vectorized text

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from nltk.tokenize import RegexpTokenizer

token = RegexpTokenizer(r'[a-zA-Z0-9]+')


cv = CountVectorizer(
    lowercase=True,
    stop_words='english',
    ngram_range=(1, 1),
    tokenizer=token.tokenize
)


X_train = cv.fit_transform(data['Clean_tweet'])
X_test = cv.transform(val_data['Clean_tweet'])

Y_train = data['Sentiment']
Y_test = val_data['Sentiment']


/usr/local/lib/python3.11/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Encoding emotions

In [ ]:
from sklearn.preprocessing import LabelEncoder


encoder = LabelEncoder()


Y_train = encoder.fit_transform(data['Sentiment'])
Y_test = encoder.transform(val_data['Sentiment'])

class_mapping = dict(zip(encoder.classes_, range(len(encoder.classes_))))
print(class_mapping)

{'Irrelevant': 0, 'Negative': 1, 'Neutral': 2, 'Positive': 3}


**Model Building**

MultinomialNB

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn import metrics
MNB_classifier = MultinomialNB().fit(X_train, Y_train)
predicted_MNB = MNB_classifier.predict(X_test)
print("Accuracy of MNB Classifier:", metrics.accuracy_score(Y_test, predicted_MNB))
print("\nClassification Report:\n", metrics.classification_report(Y_test, predicted_MNB))

Accuracy of MNB Classifier: 0.829

Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.77      0.83       172
           1       0.76      0.89      0.82       266
           2       0.92      0.73      0.82       285
           3       0.80      0.91      0.85       277

    accuracy                           0.83      1000
   macro avg       0.85      0.82      0.83      1000
weighted avg       0.84      0.83      0.83      1000



Random Forest Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier
RF_classifier = RandomForestClassifier().fit(X_train, Y_train)
predicted_RF = RF_classifier.predict(X_test)
print("Accuracy of RF Classifier:", metrics.accuracy_score(Y_test, predicted_RF))
print("Classification Report:\n", metrics.classification_report(Y_test, predicted_RF))

Accuracy of RF Classifier: 0.961
Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.97      0.97       172
           1       0.95      0.97      0.96       266
           2       0.96      0.94      0.95       285
           3       0.96      0.97      0.97       277

    accuracy                           0.96      1000
   macro avg       0.96      0.96      0.96      1000
weighted avg       0.96      0.96      0.96      1000



SVC Classifier

In [ ]:
from sklearn.svm import SVC
SVC_classifier = SVC().fit(X_train, Y_train)
predicted_SVC= SVC_classifier.predict(X_test)
print("Accuracy of SVC Classifier:", metrics.accuracy_score(Y_test, predicted_SVC))
print("Classification Report:\n", metrics.classification_report(Y_test, predicted_SVC))

Accuracy of SVC Classifier: 0.951
Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.94      0.96       172
           1       0.94      0.96      0.95       266
           2       0.97      0.94      0.95       285
           3       0.93      0.96      0.94       277

    accuracy                           0.95      1000
   macro avg       0.95      0.95      0.95      1000
weighted avg       0.95      0.95      0.95      1000



**Hyperparameter Tuning**

Multinomial Naïve Bayes

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.naive_bayes import MultinomialNB

nb_params = {'alpha': [0.001, 0.01, 0.1, 0.2, 0.5]}
nb_model = GridSearchCV(MultinomialNB(), nb_params, cv=5, scoring='accuracy')
nb_model.fit(X_train, Y_train)
y_pred_nb = nb_model.predict(X_test)
nb_accuracy = accuracy_score(Y_test, y_pred_nb)
print(f'Best MultinomialNB Params: {nb_model.best_params_}, Accuracy: {nb_accuracy:.2f}')
print("Classification Report: （Tuned MNB)\n", metrics.classification_report(Y_test, y_pred_nb))

Best MultinomialNB Params: {'alpha': 0.5}, Accuracy: 0.85
Classification Report: （Tuned MNB)
               precision    recall  f1-score   support

           0       0.90      0.80      0.85       172
           1       0.80      0.90      0.84       266
           2       0.94      0.77      0.85       285
           3       0.81      0.91      0.86       277

    accuracy                           0.85      1000
   macro avg       0.86      0.85      0.85      1000
weighted avg       0.86      0.85      0.85      1000



Random Forest

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

rf_params = {'n_estimators': [50, 100, 200],
             'max_depth': [None, 10, 20],
             'min_samples_split': [2, 5, 10]}

rf_model_grid = GridSearchCV(RandomForestClassifier(random_state=42), rf_params, cv=5, scoring='accuracy')
rf_model_grid.fit(X_train, Y_train)

print(f'Best Random Forest Params: {rf_model_grid.best_params_}')

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

best_rf_model = RandomForestClassifier(
    n_estimators=rf_model_grid.best_params_['n_estimators'],
    max_depth=rf_model_grid.best_params_['max_depth'],
    min_samples_split=rf_model_grid.best_params_['min_samples_split'],
    random_state=42
)

best_rf_model.fit(X_train, Y_train)
y_pred_rf = best_rf_model.predict(X_test)
rf_accuracy = accuracy_score(Y_test, y_pred_rf)

print(f'Accuracy: {rf_accuracy:.2f}')
print("Classification Report: (Tuned Random Forest)\n", classification_report(Y_test, y_pred_rf))


SVM

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

svm_params = {
    'C': [0.01, 0.1, 1, 10, 100],
    'gamma': [1e-4, 1e-3, 1e-2, 0.1, 1, 10],
    'kernel': ['linear', 'rbf']
}
svm_model = GridSearchCV(SVC(), svm_params, cv=5, scoring='accuracy')
svm_model.fit(X_train, Y_train)
print(f'Best SVC Params: {svm_model.best_params_}')

In [ ]:
y_pred_svm = svm_model.predict(X_test)
svm_accuracy = accuracy_score(Y_test, y_pred_svm)
print("Classification Report: \n", metrics.classification_report(Y_test, y_pred_svm))